# Motor Insurance Pricing — Data Exploration

**Dataset:** French Motor Third-Party Liability (freMTPL2freq / freMTPL2sev)

**Goal of this notebook:** This notebook explores the French Motor Third-Party Liability dataset to understand:
- The structure of the data
- The distribution of claims
- Potential risk factors affecting claim frequency
- Data quality issues before modelling

## 0. Setup

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_DIR = "../data/raw"

## 1. Load and Inspect

Load both the frequency and severity tables, check their shape, dtypes, and how they relate via `IDpol`.

In [12]:
freq = pd.read_csv(f"{DATA_DIR}/freMTPL2freq.csv")
sev = pd.read_csv(f"{DATA_DIR}/freMTPL2sev.csv")

print("freq shape:", freq.shape)
print("sev shape:", sev.shape)

freq shape: (678013, 12)
sev shape: (26639, 2)


In [13]:
freq.head()

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0,1,0.10,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3.0,1,0.77,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5.0,1,0.75,6,2,52,50,B12,Diesel,B,54,Picardie
3,10.0,1,0.09,7,0,46,50,B12,Diesel,B,76,Aquitaine
4,11.0,1,0.84,7,0,46,50,B12,Diesel,B,76,Aquitaine


In [14]:
freq.info()

<class 'pandas.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       678013 non-null  float64
 1   ClaimNb     678013 non-null  int64  
 2   Exposure    678013 non-null  float64
 3   VehPower    678013 non-null  int64  
 4   VehAge      678013 non-null  int64  
 5   DrivAge     678013 non-null  int64  
 6   BonusMalus  678013 non-null  int64  
 7   VehBrand    678013 non-null  str    
 8   VehGas      678013 non-null  str    
 9   Area        678013 non-null  str    
 10  Density     678013 non-null  int64  
 11  Region      678013 non-null  str    
dtypes: float64(2), int64(6), str(4)
memory usage: 62.1 MB


In [15]:
sev.head()

,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


In [18]:
sev.info()

<class 'pandas.DataFrame'>
RangeIndex: 26639 entries, 0 to 26638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   IDpol        26639 non-null  int64  
 1   ClaimAmount  26639 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 416.4 KB


In [17]:
# How many policies in freq actually have a matching claim record in sev?
matched = sev["IDpol"].isin(freq["IDpol"]).sum()
print(f"{matched} / {len(sev)} severity rows match a policy in frequency")

26444 / 26639 severity rows match a policy in frequency


#### Findings: 
- **Frequency** dataset contains policy-level information and the number of claims made.  
- **Severity** dataset contains information about the cost of individual claims.  
  
- The frequency dataset consists of **678013** rows and **12** columns 
- The severity dataset consists of **26639** rows and **2** columns   
   
The columns for the frequency and severity datasets are explained in the *CASdatasets Manual* (see README) and highlighted below as well as their data types:  
   
Frequency:  
- **IDpol** (float): The policy ID (used to link with the claims dataset). 
- **ClaimNb** (integer): Number of claims during the exposure period. 
- **Exposure** (float): The period of exposure for a policy, in years. 
- **VehPower** (integer) The power of the car (ordered values). 
- **VehAge** (integer): The vehicle age, in years. 
- **DrivAge** (integer): The driver age, in years. 
- **BonusMalus** (integer): Bonus/malus, between 50 and 350: <100 means bonus, >100 means malus in France. 
- **VehBrand** (string): The car brand (unknown categories)
- **VehGas** (string): The car gas, Diesel or regular
- **Area** (string): The density value of the city community where the car driver lives in: from "A" for rural area to "F" for urban centre
- **Density** (integer): The density of inhabitants (number of inhabitants per square-kilometer) of the city where the car driver lives in
- **Region** (string): The policy region in France (based on the 1970-2015 classification)  
  
Severity:  
- **IDpol** (integer): The occurence date (used to link with the contract dataset).
- **ClaimAmount** (float): The cost of the claim, seen as at a recent date.  
  
**195 claims** (~0.732% of severity records) reference a policy ID not present in the frequency table — a known inconsistency in this dataset. These will be excluded from the frequency-severity join going forward as it is less than 1% of claims.


## 2. Data quality

Check for missing values, duplicates, and implausible ranges (e.g. `Exposure` should be within [0, 1], ages should be sensible).

In [ ]:
print("Missing values (freq):")
print(freq.isna().sum())
print("\nDuplicate IDpol rows:", freq["IDpol"].duplicated().sum())

In [ ]:
freq.describe()

In [ ]:
# Sanity checks
print("Exposure out of [0,1]:", ((freq["Exposure"] < 0) | (freq["Exposure"] > 1)).sum())
print("DrivAge range:", freq["DrivAge"].min(), "-", freq["DrivAge"].max())
print("VehAge range:", freq["VehAge"].min(), "-", freq["VehAge"].max())

*Findings: (note any issues found here — e.g. capped exposures, extreme ages, known data quirks — and how you plan to handle them later.)*

## 3. Univariate analysis

Distributions of the target(s) and key features, one at a time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(freq["ClaimNb"], discrete=True, ax=axes[0])
axes[0].set_title("Distribution of ClaimNb")

sns.histplot(sev["ClaimAmount"], bins=50, ax=axes[1])
axes[1].set_title("Distribution of ClaimAmount")
axes[1].set_xlim(0, sev["ClaimAmount"].quantile(0.99))
plt.tight_layout()

In [ ]:
numeric_cols = ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(freq[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
categorical_cols = ["Area", "VehBrand", "VehGas", "Region"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, categorical_cols):
    freq[col].value_counts().plot(kind="bar", ax=ax)
    ax.set_title(col)
plt.tight_layout()

*Findings: (which features are skewed, which categories dominate, anything surprising.)*

## 4. Bivariate analysis

How claim frequency and severity vary with key features.

In [ ]:
freq["DrivAgeBand"] = pd.cut(freq["DrivAge"], bins=[18, 25, 35, 45, 55, 65, 100])

band_summary = freq.groupby("DrivAgeBand", observed=True).agg(
    claim_nb=("ClaimNb", "sum"),
    exposure=("Exposure", "sum")
)
band_summary["frequency"] = band_summary["claim_nb"] / band_summary["exposure"]
band_summary["frequency"].plot(kind="bar", figsize=(8, 4), title="Claim frequency by driver age band")
plt.ylabel("Claims per unit exposure")
plt.tight_layout()

In [ ]:
region_summary = freq.groupby("Region", observed=True).agg(
    claim_nb=("ClaimNb", "sum"),
    exposure=("Exposure", "sum")
)
region_summary["frequency"] = region_summary["claim_nb"] / region_summary["exposure"]
region_summary["frequency"].sort_values().plot(kind="barh", figsize=(8, 6), title="Claim frequency by region")
plt.xlabel("Claims per unit exposure")
plt.tight_layout()

*Findings: (which features look like real risk drivers vs noise.)*

## 5. Correlations

Check for multicollinearity among numeric features.

In [ ]:
corr = freq[numeric_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix — numeric features")
plt.tight_layout()

*Findings: (note any strongly correlated pairs, e.g. DrivAge vs BonusMalus, and what that implies for modeling later.)*

## 6. Exposure-aware frequency

Reminder: `Exposure` is the fraction of the year a policy was active. Always compute `ClaimNb / Exposure` (or aggregate sums as above) rather than comparing raw claim counts across policies with different exposure — otherwise short-exposure policies look artificially safe or risky.

In [ ]:
overall_frequency = freq["ClaimNb"].sum() / freq["Exposure"].sum()
print(f"Overall portfolio claim frequency: {overall_frequency:.4f} claims per policy-year")

## 7. Summary of findings

*(Write a short recap here: what stood out, which features look predictive, any data quality issues to address in the next notebook, and what you'll do next — e.g. cleaning, encoding, and building frequency/severity models.)*